# P6 검산 오류 로그 수집

**오류가 난 기존 Colab 노트북에 아래 코드 셀 두 개를 복사해 실행하세요.** 같은 런타임의 `/content/MI_P6_r1`과 Drive 결과를 사용합니다. 학습·설치·입력 준비 셀은 다시 실행할 필요가 없습니다.

이 진단은 학습 코드·계약·checkpoint를 수정하지 않습니다. 기존 검산 명령의 stdout/stderr를 화면과 새 로그에 함께 저장합니다. 검산 자체가 통과하면 원래 검산기가 새 audit 파일을 추가할 수 있습니다. traceback의 마지막 부분 또는 다운로드한 로그를 전달하세요.

In [ ]:
from pathlib import Path
import sys, json, hashlib, subprocess, time
from collections import deque

ROOT=Path('/content/MI_P6_r1')
OUTPUT=Path('/content/drive/MyDrive/boolean_interp_v1_4/P6_r1')
script=ROOT/'scripts/verify_v1_4_p6_return.py'
assert script.is_file(), '오류가 났던 기존 Colab 런타임에서 실행하세요.'
assert OUTPUT.is_dir(), '기존 Drive 연결과 P6_r1 경로를 확인하세요.'
stamp=str(time.time_ns())
LOG=OUTPUT/'diagnostics'/f'audit_failure_{stamp}.log'
LOG.parent.mkdir(parents=True,exist_ok=True)
contract_path=ROOT/'experiment_v1_4/p6_r1/contract.json'
config=json.loads(contract_path.read_text())
results={r['name']:(OUTPUT/'runs'/r['name']/'result.json').is_file() for r in config['runs']}
changed=[]
for name,digest in config['files'].items():
    path=ROOT/name
    if not path.is_file():
        changed.append({'path':name,'problem':'missing'})
    elif hashlib.sha256(path.read_bytes()).hexdigest()!=digest:
        changed.append({'path':name,'problem':'hash mismatch'})
summary={'python':sys.version,'executable':sys.executable,
         'run_results_present':sum(results.values()),'run_results_expected':len(results),
         'missing_runs':[k for k,v in results.items() if not v],
         'contract_file_mismatches':changed,
         'input_manifest_exists':(OUTPUT/'input_manifest.json').is_file(),
         'training_complete_exists':(OUTPUT/'training_complete.json').is_file()}
print(json.dumps(summary,ensure_ascii=False,indent=2))
command=[sys.executable,'-u',str(script),'--root',str(ROOT),'--output',str(OUTPUT),'--device','cuda']
tail=deque(maxlen=70)
with LOG.open('x') as log:
    log.write(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
    log.write('COMMAND '+json.dumps(command)+'\n');log.flush()
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            log.write(line);log.flush()
            print(line,end='',flush=True);tail.append(line)
        returncode=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait()
        log.write('Diagnostic audit interrupted by user.\n');raise
    log.write(f'\nRETURN CODE: {returncode}\n')
print('\n종료 코드:',returncode,'\n로그:',LOG)
if returncode:
    print('\n--- 전달할 traceback 마지막 부분 ---\n'+''.join(tail))
else:
    print('이번 검산 실행은 통과했습니다. 반환 증빙 검토 전에는 P6 완료가 아닙니다.')


In [ ]:
from google.colab import files
files.download(str(LOG))
